### Terminology time ! 

##### Trace, n. 
A **trace** is the end-to-end story of a single request as it moves through your system. For us, it's one RAG call.  

##### Span, n. 
A **span** is one operation within a trace. A trace is made of one or more spans, organized as a tree. Each span has a name, a start and end time, and a set of attributes. For us we will have one span inside the trace, but for agents one trace will have multiple spans.  

##### Attribute, n.
**Attributes** are key-value pairs attached to a span - anything you want to record, like the number of tokens used or the cost of a call.

In [11]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

#Creates the SDK's central configuration object
provider = TracerProvider() 

# Wires a processor that forwards every finished span to the console exporter one at a time
# FYI: 'Simple' means synchronous and immediate
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
) 

# Registers the provider globally so every call to `trace.get_tracer(...)` returns a tracer backed by it
trace.set_tracer_provider(provider)

# Returns a `Tracer` we use to create spans. 
tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [12]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    """
    Subclass of RAGBase that wraps key methods with OpenTelemetry spans
    for observability and tracing.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.last_call: LLMCallRecord = None

    def search(self, query, num_results=5):
        """Search with tracing span."""
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            
            results = super().search(query, num_results=num_results)
            
            span.set_attribute("results_count", len(results))
            return results

    def llm(self, prompt):
        """LLM call with tracing span."""
        with tracer.start_as_current_span("llm_call") as span:
            span.set_attribute("model", self.model)
            span.set_attribute("prompt_length", len(prompt))
            
            response = super().llm(prompt)
            usage = response.usage

            # Add response attributes if available
            if hasattr(response, 'output_text'):
                span.set_attribute("response_length", len(response.output_text))
                span.set_attribute("input_tokens", usage.input_tokens)
                span.set_attribute("output_tokens", usage.output_tokens)

            
            return response

    def rag(self, query):
        """Full RAG pipeline with tracing span."""
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            span.set_attribute("model", self.model)
            
            result = super().rag(query)
            
            span.set_attribute("result_length", len(result))
            return result

In [13]:
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from openai import OpenAI
from rag_helper import RAGBase


load_dotenv()

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()

In [14]:
rag_traced = RAGTraced(
    index=index,
    llm_client=client
)

In [15]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xfde66608c9632e00909745a598ffe5c2",
        "span_id": "0x44c962a5fce657bf",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2a04c9d74873073e",
    "start_time": "2026-07-21T19:10:59.899534Z",
    "end_time": "2026-07-21T19:10:59.901132Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "results_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "4638f3a7-f2ae-4467-9c07-40eebb2a3922",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_call",
    "context": {
       

str